In [ ]:
# Cell 1: Environment setup
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

In [ ]:
# Cell 2: Imports
import os
import torch
import numpy as np
import pandas as pd
import soundfile as sf
import torchaudio
import random
from torch.utils.data import Dataset, DataLoader

# PANNs
from models import Cnn14

In [ ]:
# Cell 3: Check GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# Cell 4: Load ASVspoof data and WhatsApp voice notes
import os

# Paths
ASVSPOOF_ROOT = os.environ.get("ASVSPOOF_ROOT", r"C:\deepfake-project\data\asvspoof")
PROTOCOL_DIR  = os.path.join(ASVSPOOF_ROOT, "ASVspoof2019_LA_cm_protocols")
TRAIN_AUDIO   = os.path.join(ASVSPOOF_ROOT, "ASVspoof2019_LA_train", "flac")
DEV_AUDIO     = os.path.join(ASVSPOOF_ROOT, "ASVspoof2019_LA_dev", "flac")

# Path to collected WhatsApp voice notes
WHATSAPP_VOICE_DIR = r"C:\whatsapp-voice-bot\ChrisKelleher1947.github.io\bot\collected_voice_notes"

# Verify paths exist
for path_name, path in [("ASVSPOOF_ROOT", ASVSPOOF_ROOT), ("PROTOCOL_DIR", PROTOCOL_DIR), 
                         ("TRAIN_AUDIO", TRAIN_AUDIO), ("DEV_AUDIO", DEV_AUDIO)]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path_name} does not exist: {path}")

if not os.path.exists(WHATSAPP_VOICE_DIR):
    print(f"WhatsApp voice directory not found: {WHATSAPP_VOICE_DIR}")
    print("Proceeding with ASVspoof data only")
 
# Read train and dev protocol files
train_df = pd.read_csv(
    os.path.join(PROTOCOL_DIR, "ASVspoof2019.LA.cm.train.trn.txt"),
    sep=" ", header=None,
    names=["speaker", "file_id", "env", "attack", "label"]
)
 
dev_df = pd.read_csv(
    os.path.join(PROTOCOL_DIR, "ASVspoof2019.LA.cm.dev.trl.txt"),
    sep=" ", header=None,
    names=["speaker", "file_id", "env", "attack", "label"]
)
 
# Convert labels to integers — 0 = bonafide, 1 = spoof
train_df["label"] = train_df["label"].map({"bonafide": 0, "spoof": 1})
dev_df["label"]   = dev_df["label"].map({"bonafide": 0, "spoof": 1})
 
# Add the full file path for each audio file
train_df["path"] = train_df["file_id"].apply(lambda x: os.path.join(TRAIN_AUDIO, f"{x}.flac"))
dev_df["path"]   = dev_df["file_id"].apply(lambda x: os.path.join(DEV_AUDIO,   f"{x}.flac"))

# Add WhatsApp voice notes if directory exists
if os.path.exists(WHATSAPP_VOICE_DIR):
    whatsapp_files = [f for f in os.listdir(WHATSAPP_VOICE_DIR) if f.endswith('.ogg')]
    
    whatsapp_df = pd.DataFrame({
        "speaker": ["chris"] * len(whatsapp_files),
        "file_id": [os.path.splitext(f)[0] for f in whatsapp_files],
        "env": ["whatsapp"] * len(whatsapp_files),
        "attack": ["-"] * len(whatsapp_files),
        "label": [0] * len(whatsapp_files),
        "path": [os.path.join(WHATSAPP_VOICE_DIR, f) for f in whatsapp_files]
    })
    
    # Merge with training data
    train_df_original_count = len(train_df)
    train_df = pd.concat([train_df, whatsapp_df], ignore_index=True)
    
    print(f"  Added {len(whatsapp_df)} WhatsApp voice notes to training set")
    print(f"  ASVspoof samples: {train_df_original_count}")
    print(f"  WhatsApp samples: {len(whatsapp_df)}")
    print(f"  Total:            {len(train_df)}")
else:
    print("No WhatsApp voice notes added")

print(f"\nFinal dataset:")
print(f"Training samples:   {len(train_df)}")
print(f"Dev samples:        {len(dev_df)}")
print(f"Train label split:  {train_df['label'].value_counts().to_dict()}")
print(f"Dev label split:    {dev_df['label'].value_counts().to_dict()}")

In [ ]:
# Cell 5: Dataset with WhatsApp augmentation and logging
import os
import tempfile
import subprocess
import numpy as np
import torch
import random
import soundfile as sf
from torch.utils.data import Dataset
from scipy import signal

# Audio preprocessing configuration
SAMPLE_RATE = 16000
MAX_SAMPLES = 64000


# Dataset class responsible for audio loading and augmentation
class ASVspoofDataset(Dataset):

    # Store dataset reference and augmentation probabilities
    def __init__(self, df, augment=True):
        self.df = df
        self.augment = augment

        self.probs = {
            "opus": 0.8,
            "noise": 0.5,
            "bandpass": 0.5,
            "volume": 0.4,
            "pitch": 0.3
        }

    # Return total number of samples in dataset
    def __len__(self):
        return len(self.df)

    # Load both wav and ogg audio formats
    def load_audio(self, path):

        if path.endswith(".ogg"):
            tmp_wav = tempfile.mktemp(suffix=".wav")

            try:
                # Convert ogg audio into wav format using ffmpeg
                subprocess.run([
                    "ffmpeg", "-y", "-loglevel", "error",
                    "-i", path,
                    "-ar", str(SAMPLE_RATE),
                    "-ac", "1",
                    tmp_wav
                ], check=True)

                audio, _ = sf.read(tmp_wav, dtype="float32")
                return audio

            finally:
                # Remove temporary conversion file after processing
                if os.path.exists(tmp_wav):
                    os.remove(tmp_wav)

        audio, _ = sf.read(path, dtype="float32")
        return audio

    # Adds random background noise
    def add_noise(self, audio):
        noise = np.random.randn(len(audio)) * random.uniform(0.003, 0.02)
        return audio + noise

    # Applies random volume scaling 
    def change_volume(self, audio):
        return audio * random.uniform(0.5, 1.5)

    # Applies simple pitch shifting 
    def pitch_shift(self, audio, semitones):
        if semitones == 0:
            return audio

        rate = 2 ** (semitones / 12)
        idx = np.round(np.arange(0, len(audio), rate)).astype(int)
        idx = idx[idx < len(audio)]

        return audio[idx]

    # Applies bandpass filtering
    def bandpass_filter(self, audio, lowcut=50, highcut=400):

        nyq = SAMPLE_RATE / 2

        sos = signal.butter(
            5,
            [lowcut / nyq, highcut / nyq],
            btype="band",
            output="sos"
        )

        return signal.sosfilt(sos, audio).astype(np.float32)

    # Main augmentation pipeline
    def augment_audio(self, audio):

        # Simulate WhatsApp Opus compression
        if random.random() < self.probs["opus"]:

            tmp_in = tempfile.mktemp(suffix=".wav")
            tmp_ogg = tempfile.mktemp(suffix=".ogg")
            tmp_out = tempfile.mktemp(suffix=".wav")

            try:
                sf.write(tmp_in, audio, SAMPLE_RATE)

                bitrate = random.choice([12, 16, 24])

                subprocess.run([
                    "ffmpeg", "-y", "-loglevel", "error",
                    "-i", tmp_in,
                    "-c:a", "libopus",
                    "-b:a", f"{bitrate}k",
                    tmp_ogg
                ], check=True)

                subprocess.run([
                    "ffmpeg", "-y", "-loglevel", "error",
                    "-i", tmp_ogg,
                    "-ar", str(SAMPLE_RATE),
                    "-ac", "1",
                    tmp_out
                ], check=True)

                audio, _ = sf.read(tmp_out, dtype="float32")

            finally:
                # Remove temporary files created during augmentation
                for f in [tmp_in, tmp_ogg, tmp_out]:
                    if os.path.exists(f):
                        os.remove(f)

        # Apply random bandpass filtering
        if random.random() < self.probs["bandpass"]:
            audio = self.bandpass_filter(audio, 50, random.randint(350, 450))

        # Apply random noise injection
        if random.random() < self.probs["noise"]:
            audio = self.add_noise(audio)

        # Apply random volume adjustment
        if random.random() < self.probs["volume"]:
            audio = self.change_volume(audio)

        # Apply random pitch shifting
        if random.random() < self.probs["pitch"]:
            audio = self.pitch_shift(audio, random.uniform(-3, 3))

        return audio

    # Ensure all audio samples are fixed length
    def fix_length(self, audio):

        if len(audio) > MAX_SAMPLES:
            audio = audio[:MAX_SAMPLES]
        else:
            audio = np.pad(audio, (0, MAX_SAMPLES - len(audio)))

        return audio

    # Normalize audio amplitude before model input
    def normalize(self, audio):

        peak = np.abs(audio).max()

        return audio / peak if peak > 0 else audio

    # Main sample loading and preprocessing
    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        audio = self.load_audio(row["path"])

        # Skip augmentation for WhatsApp recordings
        if self.augment and not row["path"].endswith(".ogg"):
            audio = self.augment_audio(audio)

        audio = self.fix_length(audio)
        audio = self.normalize(audio)

        # Return processed audio tensor and corresponding label
        return {
            "input": torch.tensor(audio, dtype=torch.float32),
            "label": torch.tensor(row["label"], dtype=torch.long)
        }

In [ ]:
# Cell 6: Load model

device = torch.device("cuda")

# Initialize PANNs CNN14 architecture
model = Cnn14(
    sample_rate=16000,
    window_size=1024,
    hop_size=320,
    mel_bins=64,
    fmin=50,
    fmax=8000,
    classes_num=2
)

# Replace original AudioSet classification layer for deepfake detection
model.fc_audioset = torch.nn.Linear(2048, 2)

# Move model to GPU device
model = model.to(device)

print("PANNs CNN14 loaded")

# Calculate total number of trainable model parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

# Display parameter statistics
print(f"Trainable parameters: {trainable:,} / {total:,}")

In [ ]:
# Cell 7: Create data loaders

from torch.utils.data import WeightedRandomSampler

# Create training and validation datasets
train_dataset = ASVspoofDataset(train_df, augment=True)
dev_dataset   = ASVspoofDataset(dev_df, augment=False)

# Calculate class weights based on label frequency
class_counts  = train_df["label"].value_counts().sort_index().values
class_weights = 1.0 / class_counts

# Assign sampling weight to each training sample
sample_weights = train_df["label"].map({
    0: class_weights[0],
    1: class_weights[1]
}).values

# Oversample minority classes during training
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_dataset),
    replacement=True
)

# Create data loaders for training and validation
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    sampler=sampler
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=8,
    shuffle=False
)

# Display total number of training batches
print(f"Training batches: {len(train_loader)}")

In [ ]:
# Cell 8: Optimiser and evaluation function

from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR
from sklearn.metrics import roc_auc_score

# Configure AdamW optimiser for model training
optimiser = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4,
    weight_decay=0.01
)

# Configure linear learning rate warmup schedule
total_steps  = len(train_loader) * 5
warmup_steps = int(0.1 * total_steps)

scheduler = LinearLR(
    optimiser,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=warmup_steps
)

# Evaluation function
def evaluate(model, loader, device):

    model.eval()

    total_loss = 0
    correct = 0

    all_labels = []
    all_probs = []

    # Disable gradient tracking during evaluation
    with torch.no_grad():

        for batch in loader:

            inputs = batch["input"].to(device).to(torch.float32)
            labels = batch["label"].to(device)

            # Run model inference on validation batch
            outputs = model(inputs)

            logits = outputs["clipwise_output"]

            loss = torch.nn.functional.cross_entropy(logits, labels)

            total_loss += loss.item()

            # Convert logits into prediction probabilities
            probs = torch.softmax(logits, dim=-1)

            preds = probs.argmax(dim=-1)

            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    # Calculate validation performance metrics
    avg_loss = total_loss / len(loader)
    accuracy = correct / len(loader.dataset)
    roc_auc = roc_auc_score(all_labels, all_probs)

    return avg_loss, accuracy, roc_auc

# Display optimiser and scheduler configuration summary
print("Optimiser and evaluation function ready")
print(f"Total training steps: {total_steps}")
print(f"Warmup steps:         {warmup_steps}")

Optimiser and evaluation function ready
Total training steps: 15865
Warmup steps:         1586


In [ ]:
# Cell 9: Training loop

from torch.amp import autocast
from sklearn.metrics import roc_auc_score

# Training configuration and model checkpoint settings
EPOCHS       = 5
EVAL_STEPS   = 200
SAVE_DIR     = r"C:\deepfake-project\models\panns_cnn14"

best_roc_auc = 0.0
patience     = 0
PATIENCE_MAX = 4

# Create model save directory
os.makedirs(SAVE_DIR, exist_ok=True)

# Display training configuration summary
print("\n" + "="*60)
print("TRAINING (PANNs CNN14)")
print("="*60)
print(f"Save directory: {SAVE_DIR}")
print(f"Eval frequency: every {EVAL_STEPS} steps")
print(f"Early stopping: {PATIENCE_MAX} evaluations without improvement")
print("="*60 + "\n")

# Main training loop across all epochs
for epoch in range(EPOCHS):

    model.train()

    epoch_loss = 0
    step = 0

    # Reset augmentation statistics at the start of each epoch
    try:
        AUG_STATS.clear()
        AUG_TOTAL = 0
    except:
        pass

    print(f"\n================ EPOCH {epoch+1} START ================\n")

    # Iterate through training batches
    for batch in train_loader:

        inputs = batch["input"].to(device)
        labels = batch["label"].to(device)

        # Ensure model inputs use float32 precision
        inputs = inputs.to(torch.float32)

        # Enable mixed precision
        with autocast(device_type="cuda", dtype=torch.bfloat16):

            outputs = model(inputs)

            logits = outputs["clipwise_output"]

            loss = torch.nn.functional.cross_entropy(logits, labels)

        # Backpropagation and parameter update step
        loss.backward()

        # Prevent exploding gradients during training
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimiser.step()
        scheduler.step()
        optimiser.zero_grad()

        epoch_loss += loss.item()
        step += 1

        # Display running training loss
        if step % 50 == 0:
            avg_loss = epoch_loss / step
            print(f"Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {avg_loss:.4f}")

        # Run validation evaluation at fixed intervals
        if step % EVAL_STEPS == 0:

            dev_loss, dev_acc, dev_roc = evaluate(model, dev_loader, device)

            print(f"\n>>> Eval @ step {step} | Loss: {dev_loss:.4f} | Acc: {dev_acc:.4f} | ROC-AUC: {dev_roc:.4f}")

            # Save model checkpoint if ROC-AUC improves
            if dev_roc > best_roc_auc:

                best_roc_auc = dev_roc
                patience = 0

                torch.save(
                    model.state_dict(),
                    os.path.join(SAVE_DIR, "best_model.pt")
                )

                print(f"     New best model saved (ROC-AUC: {best_roc_auc:.4f})")

            else:
                # Increment patience counter when validation performance stalls
                patience += 1

                print(f"    No improvement — patience {patience}/{PATIENCE_MAX}")

                # Stop training early if no improvement persists
                if patience >= PATIENCE_MAX:
                    print("\nEarly stopping triggered")
                    break

            model.train()

    # Display augmentation usage statistics after each epoch
    print("\n================ AUGMENTATION STATS ================")

    try:
        print(f"Total augmented samples: {AUG_TOTAL}")

        if AUG_TOTAL > 0:

            for k, v in AUG_STATS.items():

                pct = (v / AUG_TOTAL) * 100

                print(f"{k}: {v} ({pct:.1f}%)")

        else:
            print("No augmentation stats recorded")

    except:
        print("Augmentation stats not available for PANNs pipeline")

    print("====================================================\n")

    # Display average training loss after each epoch
    print(f"\nEpoch {epoch+1} complete | Avg loss: {epoch_loss/len(train_loader):.4f}\n")

    if patience >= PATIENCE_MAX:
        break


# Display final training summary
print("\n" + "="*60)
print(f"TRAINING COMPLETE | Best ROC-AUC: {best_roc_auc:.4f}")
print(f"Best model saved to: {SAVE_DIR}")
print("="*60)